In [1]:
import { ChatGoogle } from "npm:@langchain/google";
import { z } from "npm:zod";
import { parse } from "jsr:@std/dotenv";

In [2]:
const env = parse(
  await Deno.readTextFile(".env")
);

const apiKey = env.GOOGLE_API_KEY;

console.log(
  apiKey
    ? "Gemini API Key 설정 완료"
    : "Gemini API Key가 없습니다."
);

Gemini API Key 설정 완료


Gemini 모델 생성
temperature : 보통 0~1사이의 값을 인수로 받는다. 낮을수록 가장확률이 높은 단어를 우선 선택하지만 단조로운 답변이 나올수있다. 높을수록 다양한 단어가 나오지만 환각증상이 심해질수 있다.

In [3]:
const model = new ChatGoogle({
  model: "gemini-3.1-flash-lite",
  apiKey,
  temperature: 0,
});

데이터 스키마 구성하기

In [4]:
const RestaurantSearchSchema = z.object({

  district: z
    .string()
    .nullable()
    .describe(
      "사용자가 찾고 있는 지역 또는 행정구역. 예: 남구, 수성구"
    ),

  category: z
    .string()
    .nullable()
    .describe(
      `
사용자가 원하는 음식 종류 또는 식당 유형을
아래 실제 데이터 카테고리 중 가장 적절한 값으로 변환합니다.

사용 가능한 카테고리:
- 한식
- 일식
- 중식
- 양식
- 세계요리
- 특별한 술집
- 전통차/커피전문점

사용자가 카테고리를 직접 말하지 않더라도
질문의 의미를 판단하여 가장 적절한 카테고리로 변환합니다.

변환 예시:
- 한식집, 한정식, 국밥, 찌개, 불고기, 백반 → 한식
- 초밥, 스시, 사시미, 돈카츠, 우동, 라멘 → 일식
- 짜장면, 짬뽕, 탕수육, 중국집 → 중식
- 파스타, 스테이크, 이탈리안, 브런치 → 양식
- 케밥, 카레, 인도요리, 태국요리, 베트남요리 등 → 세계요리(케밥,카레등)
- 술집, 주점, 술 한잔할 곳, 맥주 마실 곳 → 특별한 술집
- 카페, 커피숍, 커피 마실 곳, 전통찻집, 차 마실 곳 → 전통차/커피전문점

사용자의 질문만으로 어떤 카테고리인지 판단할 수 없거나
특정 음식 종류를 요구하지 않은 경우에는 null을 반환합니다.
  `
    ),

  budgetMin: z
    .number()
    .nullable()
    .describe(
      "1인당 최소 예산을 원 단위 숫자로 변환한 값"
    ),

  budgetMax: z
    .number()
    .nullable()
    .describe(
      "1인당 최대 예산을 원 단위 숫자로 변환한 값"
    ),

  peopleCount: z
    .number()
    .nullable()
    .describe(
      "식사 인원수. 사용자가 말하지 않았다면 null"
    ),

  preferences: z
    .array(z.string())
    .describe(
      "조용함, 부모님 동반, 주차, 룸 등 사용자가 원하는 추가 조건"
    )

});

// 스키마에서 타입을 자동으로 추출
type RestaurantSearch =
  z.infer<
    typeof RestaurantSearchSchema
  >;

답변에 스키마 적용하기

In [5]:
const structuredModel =
  model.withStructuredOutput(
    RestaurantSearchSchema
  );

자연어 질문을 구조화하여 변경하기

In [6]:
const query =
  "대구 남구에서 부모님 모시고 갈 조용한 한식집 찾아줘. 1인 3만원 정도로.";
  const searchCondition =
  await structuredModel.invoke(query);

console.log(searchCondition);

{
  district: "남구",
  category: "한식",
  budgetMin: 30000,
  budgetMax: 30000,
  peopleCount: null,
  preferences: [ "부모님 동반", "조용함" ]
}


In [7]:
const query2 =
  "2만원 안팎으로 가볍게 먹을 만한 곳";

const result2 =
  await structuredModel.invoke(query2);

console.log(result2);

{
  district: null,
  category: null,
  budgetMin: 15000,
  budgetMax: 25000,
  peopleCount: null,
  preferences: []
}


In [8]:
//예산 계산 함수
function calculateTotalBudget(
  budget: number | null,
  peopleCount: number | null
): number | null {

  // 금액이 없으면 계산 불가능
  if (budget === null) {
    return null;
  }

  // 인원수가 없으면 기본값 1명
  const count = peopleCount ?? 1;

  return budget * count;
}

In [9]:
// 재사용 가능 함수로 만들기
async function parseRestaurantQuery(
  query: string
):Promise<RestaurantSearch> {
  return await structuredModel.invoke(query);
}

In [10]:
const result =
  await parseRestaurantQuery(
    "대구 남구에서 부모님 모시고 갈 조용한 한식집 찾아줘. 1인 3만원 정도로."
  );

console.log(result);

{
  district: "남구",
  category: "한식",
  budgetMin: 30000,
  budgetMax: 30000,
  peopleCount: null,
  preferences: [ "부모님 동반", "조용함" ]
}
